In [1]:
import joblib
import numpy as np
import os

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

In [2]:
save_dir = "data/processed"

raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
temperature = np.load(os.path.join(save_dir, "temperature.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
scaler      = joblib.load(os.path.join(save_dir, "scaler.pkl"))

num_train_samples = len(train_idx)
num_val_samples   = len(val_idx)
num_test_samples  = len(test_idx)

temp_mean = scaler.mean_[1]
temp_std  = scaler.scale_[1]

print(f"raw_data:    {raw_data.shape}")
print(f"temperature: {temperature.shape}")
print(f"train/val/test: {num_train_samples} / {num_val_samples} / {num_test_samples}")

temperature_normalized = (temperature - temp_mean) / temp_std

raw_data:    (420451, 14)
temperature: (420451,)
train/val/test: 210225 / 105112 / 105114


In [3]:
sampling_rate   = 6
sequence_length = 120
delay           = sampling_rate * (sequence_length + 24 - 1)
batch_size      = 256

class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [4]:
train_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train_samples,
)

val_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples, end_index=num_train_samples + num_val_samples,
)

test_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples + num_val_samples, end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# Quick sanity check
for inputs, targets in train_loader:
    print("Input shape:", inputs.shape)   # (batch_size, sequence_length, num_features)
    print("Target shape:", targets.shape) # (batch_size,)
    break

Input shape: torch.Size([256, 120, 14])
Target shape: torch.Size([256])


In [5]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    total_mae  = 0.0
    n          = 0
    with torch.set_grad_enabled(training):
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds = model(inputs)
            loss  = criterion(preds, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            total_mae  += torch.sum(torch.abs(preds - targets)).item()
            n          += inputs.size(0)
   # mae_celsius = (total_mae / n) * temp_std
    return total_loss / n, total_mae / n   # MAE already in °C


def get_predictions(model, dataset):
    model.eval()
    all_preds     = []
    all_targets   = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    with torch.no_grad():
        for inputs, targets in loader:
            preds = model(inputs.to(device)).cpu().numpy()
            all_preds.append(preds * temp_std + temp_mean)  # un-normalize predictions
            all_targets.append(targets.numpy())              # targets already in °C
    return np.concatenate(all_preds), np.concatenate(all_targets)

In [6]:
class ModelCheckpoint:
    """Saves the best model based on a monitored metric."""
    def __init__(self, filepath, monitor="val_mae", mode="min", verbose=True):
        self.filepath = filepath
        self.monitor  = monitor
        self.verbose  = verbose
        self.best     = float("inf") if mode == "min" else float("-inf")
        self.mode     = mode

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = value < self.best if self.mode == "min" else value > self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  ✓ Best model saved ({self.monitor}: {value:.2f}°C)")
        return improved


class EarlyStopping:
    """Stops training when a monitored metric stops improving."""
    def __init__(self, monitor="val_mae", patience=5, min_delta=1e-4, mode="min"):
        self.monitor     = monitor
        self.patience    = patience
        self.min_delta   = min_delta
        self.mode        = mode
        self.best        = float("inf") if mode == "min" else float("-inf")
        self.counter     = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == "min"
                    else value > self.best + self.min_delta)
        if improved:
            self.best    = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  Early stopping triggered (no improvement for {self.patience} epochs)")
        return improved


class ReduceLROnPlateau:
    """Wraps PyTorch scheduler with the same callback interface."""
    def __init__(self, optimizer, monitor="val_mae", patience=3, factor=0.5):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=patience, factor=factor
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])

In [7]:
class SimpleRNNModel(nn.Module):
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        self.rnn  = nn.RNN(input_size=num_features, hidden_size=hidden_size,
                           batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # out: (batch, seq_len, hidden_size)
        # return_sequences=False equivalent → take last timestep
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)

In [16]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = SimpleRNNModel(num_features=raw_data.shape[-1], hidden_size = 128).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_simple_rnn")

callbacks = [
    ModelCheckpoint("jena_simple_rnn_best.pt", monitor="val_mae"),
]

# --- Training loop ---
epochs = 10
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f}°C | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}°C")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

writer.close()

Epoch 01 — train loss: 25.0929, train MAE: 3.62°C | val loss: 9.4864, val MAE: 2.41°C
  ✓ Best model saved (val_mae: 2.41°C)
Epoch 02 — train loss: 10.6341, train MAE: 2.55°C | val loss: 9.3769, val MAE: 2.41°C
  ✓ Best model saved (val_mae: 2.41°C)
Epoch 03 — train loss: 10.2929, train MAE: 2.50°C | val loss: 8.9169, val MAE: 2.32°C
  ✓ Best model saved (val_mae: 2.32°C)
Epoch 04 — train loss: 9.8005, train MAE: 2.44°C | val loss: 8.7927, val MAE: 2.30°C
  ✓ Best model saved (val_mae: 2.30°C)
Epoch 05 — train loss: 9.2638, train MAE: 2.38°C | val loss: 9.1704, val MAE: 2.37°C
Epoch 06 — train loss: 9.2888, train MAE: 2.38°C | val loss: 8.9466, val MAE: 2.33°C
Epoch 07 — train loss: 8.7743, train MAE: 2.32°C | val loss: 9.5728, val MAE: 2.41°C
Epoch 08 — train loss: 8.6848, train MAE: 2.31°C | val loss: 9.2731, val MAE: 2.37°C
Epoch 09 — train loss: 8.0907, train MAE: 2.23°C | val loss: 9.3430, val MAE: 2.37°C
Epoch 10 — train loss: 8.0396, train MAE: 2.22°C | val loss: 9.5480, val MAE

In [17]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("jena_simple_rnn_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae:.2f}°C")


Test MAE: 2.45°C


In [10]:
class StackedRNNModel(nn.Module):
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        self.rnn = nn.RNN(input_size=num_features, hidden_size=hidden_size,
                  num_layers=2, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x, _   = self.rnn(x)    
        return self.head(x[:, -1, :]).squeeze(-1)

In [11]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = StackedRNNModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_stacked_rnn")

callbacks = [
    ModelCheckpoint("jena_stacked_rnn_best.pt", monitor="val_mae"),
]

# --- Training loop ---
epochs = 10
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f}°C | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}°C")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

writer.close()

Epoch 01 — train loss: 55.7278, train MAE: 5.51°C | val loss: 24.1126, val MAE: 3.58°C
  ✓ Best model saved (val_mae: 3.58°C)
Epoch 02 — train loss: 17.0189, train MAE: 3.08°C | val loss: 12.7250, val MAE: 2.66°C
  ✓ Best model saved (val_mae: 2.66°C)
Epoch 03 — train loss: 12.0315, train MAE: 2.67°C | val loss: 10.3010, val MAE: 2.46°C
  ✓ Best model saved (val_mae: 2.46°C)
Epoch 04 — train loss: 10.7487, train MAE: 2.55°C | val loss: 9.5267, val MAE: 2.38°C
  ✓ Best model saved (val_mae: 2.38°C)
Epoch 05 — train loss: 10.2412, train MAE: 2.49°C | val loss: 9.3490, val MAE: 2.37°C
  ✓ Best model saved (val_mae: 2.37°C)
Epoch 06 — train loss: 9.9469, train MAE: 2.46°C | val loss: 9.2118, val MAE: 2.35°C
  ✓ Best model saved (val_mae: 2.35°C)
Epoch 07 — train loss: 9.7371, train MAE: 2.43°C | val loss: 9.1639, val MAE: 2.35°C
Epoch 08 — train loss: 9.5914, train MAE: 2.41°C | val loss: 8.9898, val MAE: 2.32°C
  ✓ Best model saved (val_mae: 2.32°C)
Epoch 09 — train loss: 9.6160, train MA

In [12]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("jena_stacked_rnn_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae:.2f}°C")


Test MAE: 2.51°C


In [13]:
class BidirectionalRNNModel(nn.Module):
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        self.rnn  = nn.LSTM(input_size=num_features, hidden_size=hidden_size,
                           batch_first=True, bidirectional=True)
        # bidirectional doubles the output size
        self.head = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)

In [14]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = BidirectionalRNNModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_bidirectional_rnn")

callbacks = [
    ModelCheckpoint("jena_bidirectional_rnn_best.pt", monitor="val_mae"),
    EarlyStopping(monitor="val_mae", patience=7),
    ReduceLROnPlateau(optimizer, monitor="val_mae", patience=3, factor=0.5),
]

# --- Training loop ---
epochs = 50
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f}°C | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}°C")

    for cb in callbacks:
        cb.step(metrics, model)

    if any(isinstance(cb, EarlyStopping) and cb.should_stop for cb in callbacks):
        print(f"  Stopped at epoch {epoch}")
        break

writer.close()

Epoch 01 — train loss: 135.8624, train MAE: 9.65°C | val loss: 107.7365, val MAE: 8.46°C
  ✓ Best model saved (val_mae: 8.46°C)
Epoch 02 — train loss: 83.9336, train MAE: 7.36°C | val loss: 67.8485, val MAE: 6.59°C
  ✓ Best model saved (val_mae: 6.59°C)
Epoch 03 — train loss: 55.9912, train MAE: 5.91°C | val loss: 44.9419, val MAE: 5.26°C
  ✓ Best model saved (val_mae: 5.26°C)
Epoch 04 — train loss: 39.0180, train MAE: 4.82°C | val loss: 30.7919, val MAE: 4.22°C
  ✓ Best model saved (val_mae: 4.22°C)
Epoch 05 — train loss: 28.4892, train MAE: 4.03°C | val loss: 22.5617, val MAE: 3.54°C
  ✓ Best model saved (val_mae: 3.54°C)
Epoch 06 — train loss: 22.2229, train MAE: 3.52°C | val loss: 18.0706, val MAE: 3.17°C
  ✓ Best model saved (val_mae: 3.17°C)
Epoch 07 — train loss: 18.7014, train MAE: 3.24°C | val loss: 15.7452, val MAE: 2.99°C
  ✓ Best model saved (val_mae: 2.99°C)
Epoch 08 — train loss: 16.6150, train MAE: 3.07°C | val loss: 14.0854, val MAE: 2.85°C
  ✓ Best model saved (val_mae

In [15]:
model.load_state_dict(torch.load("jena_bidirectional_rnn_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae:.2f}°C")


Test MAE: 2.54°C
